In [15]:
from langchain_core.documents import Document


In [16]:
doc=Document(
    page_content="this is the main content i am using for rag",
    metadata={
        "source":"example.txt",
        "pages":1,
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1}, page_content='this is the main content i am using for rag')

In [17]:
## create simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [18]:
sample_texts = {
    "../data/text_files/python_intro.txt": """Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",

    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems

"""
}
for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [19]:
###Textloader
from langchain_community.document_loaders import TextLoader
loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
loader
documents=loader.load()
print(documents)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [20]:
## Directory Loader
from langchain_community.document_loaders import DirectoryLoader
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'},
    show_progress=False
)
documents=dir_loader.load()
documents


[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular prog

In [21]:
from langchain_community.document_loaders.excel import UnstructuredExcelLoader
loader = UnstructuredExcelLoader(
    "../data/Excel/dummy_excel_loader_practice.xlsx",
    mode="elements"
)

docs = loader.load()

print(docs)

Thank you for using Unstructured Open Source! Try Unstructured's commercial offering for improved output quality. 10,000 free pages to start. Learn more: https://transform.unstructured.io/?utm_source=unstructured_oss Hide this notice: UNSTRUCTURED_DISABLE_NOTICE=1


[Document(metadata={'source': '../data/Excel/dummy_excel_loader_practice.xlsx', 'file_directory': '../data/Excel', 'filename': 'dummy_excel_loader_practice.xlsx', 'last_modified': '2026-11-19T12:53:25', 'page_name': 'Employees', 'page_number': 1, 'text_as_html': '<table><tr><td>Employee_ID</td><td>Name</td><td>Department</td><td>Job_Title</td><td>City</td><td>Joining_Date</td><td>Salary</td><td>Status</td></tr><tr><td>101</td><td>Aarav Sharma</td><td>IT</td><td>Python Developer</td><td>Pune</td><td>2024-01-15 00:00:00</td><td>45000</td><td>Active</td></tr><tr><td>102</td><td>Priya Patil</td><td>HR</td><td>HR Executive</td><td>Mumbai</td><td>2023-08-10 00:00:00</td><td>38000</td><td>Active</td></tr><tr><td>103</td><td>Rohan Deshmukh</td><td>IT</td><td>Java Developer</td><td>Pune</td><td>2022-11-21 00:00:00</td><td>52000</td><td>Active</td></tr><tr><td>104</td><td>Sneha Kulkarni</td><td>Finance</td><td>Accountant</td><td>Nashik</td><td>2021-06-05 00:00:00</td><td>41000</td><td>Active</td

In [22]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def process_all_pdfs(pdf_directory):
    """Load every PDF file under the supplied directory."""
    all_pdf_documents = []
    pdf_dir = Path(pdf_directory)

    # Find PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            pdf_documents = loader.load()

            # Keep source details in each page's metadata
            for doc in pdf_documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_pdf_documents.extend(pdf_documents)
            print(f"  Loaded {len(pdf_documents)} pages")

        except Exception as e:
            print(f"  Error loading {pdf_file.name}: {e}")

    print(f"\\nTotal PDF pages loaded: {len(all_pdf_documents)}")
    return all_pdf_documents


# Load all PDFs from Data_Ingestion_1/data/pdf
all_pdf_documents = process_all_pdfs("../data/pdf")

# Combine documents already loaded above:
# `documents` comes from DirectoryLoader (TXT files)
# `docs` comes from UnstructuredExcelLoader (Excel file)
all_documents = list(documents) + list(docs) + list(all_pdf_documents)

print(f"Total documents before chunking: {len(all_documents)}")

Found 1 PDF files to process
\nProcessing: Python.pdf
  Loaded 26 pages
\nTotal PDF pages loaded: 26
Total documents before chunking: 30


In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=500, chunk_overlap=50):
    """
    Split LangChain Document objects into smaller chunks.

    Args:
        documents: List of LangChain Document objects.
        chunk_size: Maximum characters in each chunk.
        chunk_overlap: Number of overlapping characters between chunks.

    Returns:
        List of chunked Document objects, with metadata preserved.
    """
    if not documents:
        print("No documents were provided for chunking.")
        return []

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\\n\\n", "\\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("\\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


chunks = split_documents(all_documents, chunk_size=500, chunk_overlap=50)
print(f"Total chunks created: {len(chunks)}")

Split 30 documents into 54 chunks
\nExample chunk:
Content: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing...
Metadata: {'source': '..\\data\\text_files\\machine_learning.txt'}
Total chunks created: 54


Embedding Stage

In [24]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [25]:
class EmbeddingManager:
    def __init__(self,model_name: str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        try:
            print(f"Loading embedding model:{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"model loaded successfully. Embedding dimension:{self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise
    def generate_embedding(self,text:List[str])->np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embedding for {len(text)} text...")
        embeddings=self.model.encode(text,show_progress_bar=True)
        print(f"Generated embedding with shape:{embeddings.shape}")
        return embeddings
    def get_embedding_dimension(self)->int:
        if not self.model:
            raise ValueError("model not loaded")
        return self.model.get_embedding_dimension()
##initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model:all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4332.18it/s]


model loaded successfully. Embedding dimension:384


In [34]:
###vector store
import os
class VectorStore:
    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            
            )
            print(f"Vector store initialize. collection :{self.collection_name}")
            print(f"Existing documents in collection:{self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store:{e}")
            raise
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError
        print(f"Adding {len(documents)} documents to vector store...")
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection:{self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        
vectorstore=VectorStore()
vectorstore

Vector store initialize. collection :pdf_documents
Existing documents in collection:0


In [35]:
chunks

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications'),
 Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for

In [36]:
###csonvert the text to embedding
texts=[doc.page_content for doc in chunks]
texts

['Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications',
 'through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems',
 'Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compati

In [37]:
embeddings=embedding_manager.generate_embedding(texts)
###stire in vector database
vectorstore.add_documents(chunks,embeddings)

Generating embedding for 54 text...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.66it/s]


Generated embedding with shape:(54, 384)
Adding 54 documents to vector store...
successfully added 54 documents to vector store
Total documents in collection:54


Retriever Pipeline From VectorStore

In [41]:
class RAGRetriever:
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager
    def retrieve(self,query:str,top_k:int=5,score_threshold:float=0.0)->List[Dict[str,Any]]:
        print(f"Retrieving documents for query:'{query}'")
        print(f"Top K:{top_k},Score threshold:{score_threshold}")
        #generating query embedding
        query_embedding=self.embedding_manager.generate_embedding([query])[0]
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs=[]
            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]

                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    similarity_score=1-distance

                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            'ids':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1

                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval :{e}")
            return []
rag_retriever=RAGRetriever(vectorstore,embedding_manager)


In [42]:
rag_retriever

In [43]:
rag_retriever.retrieve("what is python is about")

Retrieving documents for query:'what is python is about'
Top K:5,Score threshold:0.0
Generating embedding for 1 text...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.18it/s]

Generated embedding with shape:(1, 384)
Retrieved 3 documents (after filtering)


[{'ids': 'doc_6b91e40b_2',
  'content': 'Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular programming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.',
  'metadata': {'source': '..\\data\\text_files\\python_intro.txt',
   'content_length': 489,
   'doc_index': 2},
  'similarity_score': 0.6310877799987793,
  'distance': 0.3689122200012207,
  'rank': 1},
 {'ids': 'doc_1c26a456_8',
  'content': 'M o s t  A s k e d\nP y t h o n\nPython\nInterview Questions',
  'metadata': {'creationdate': '',
   'page_label': '1',
   'total_pages': 26,
   'doc_index': 8,
   'page': 0,
   'content_length': 57,
   'producer